# 02 — Daily NOAA update
**Only this notebook should be scheduled.** Existing setup: run cleaned 03 once before testing this job.
Fresh workspace: run 01, then 03, then 02. Run with maximum concurrency 1.

Current/previous year files are validated then replaced per station-year. Other station-years
remain intact. Some station updates can succeed before another fails. A failed validation
does not replace that station-year. Full downloads and a per-run manifest are retained;
there is no automatic raw-file cleanup or historical first-seen reconstruction.

NOAA schema changes and legitimate reductions in source records stop for review.
New-year files may not yet exist. Do not fill failed downloads with empty records.
Observation freshness thresholds monitor this archive, not suitability for live predictions.

## Configuration and dependencies

In [0]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
import re
import json
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

TARGET = "workspace.default.noaa_weather_raw"
REGISTRY = "workspace.default.noaa_station_registry"

# Daily updates depend on completed setup, not a hidden bootstrap copy.
for required in [TARGET, REGISTRY, "workspace.default.citibike_noaa_mapping"]:
    if not spark.catalog.tableExists(required):
        raise ValueError(f"Missing {required}. Complete 01 and 03 first.")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
run_folder = Path(
    f"/Volumes/workspace/default/citibike/noaa/download_runs/{run_id}"
)
run_folder.mkdir(parents=True, exist_ok=True)

current_year = datetime.now(timezone.utc).year

# Refresh current and previous year; retain older years in the database.
years = range(max(2026, current_year - 1), current_year + 1)

station_ids = [
    row.noaa_station_id
    for row in spark.table(REGISTRY)
        .filter("download_enabled = true")
        .select("noaa_station_id")
        .distinct()
        .collect()
]

if not station_ids:
    raise ValueError("No stations are enabled in the registry.")

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)))

print("Run:", run_id)
print("Stations:", len(station_ids))
print("Years:", list(years))

manifest = []
manifest_path = run_folder / "manifest.json"

## Download, validate, and replace station-year records

In [0]:
failures = []

for station_id in sorted(station_ids):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", station_id):
        raise ValueError(f"Unexpected station ID: {station_id}")

    for year in years:
        url = (
            "https://www.ncei.noaa.gov/oa/"
            "global-historical-climatology-network/hourly/access/"
            f"by-year/{year}/parquet/GHCNh_{station_id}_{year}.parquet"
        )

        path = run_folder / f"GHCNh_{station_id}_{year}.parquet"
        partial = path.with_suffix(".part")

        condition = (
            f"STATION = '{station_id}' "
            f"AND observed_at_utc >= TIMESTAMP '{year}-01-01 00:00:00' "
            f"AND observed_at_utc < TIMESTAMP '{year + 1}-01-01 00:00:00'"
        )

        record = {
            "station_id": station_id, "year": year, "source_url": url,
            "run_id": run_id, "local_path": str(path),
            "started_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        try:
            with session.get(
                url, stream=True, timeout=(15, 120)
            ) as response:
                response.raise_for_status()
                record["http_last_modified"] = response.headers.get("Last-Modified")
                record["http_etag"] = response.headers.get("ETag")
                with partial.open("wb") as output:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            output.write(chunk)

            if partial.stat().st_size < 8:
                raise ValueError("Downloaded file is too small.")

            with partial.open("rb") as file:
                header = file.read(4)
                file.seek(-4, 2)
                footer = file.read(4)

            if header != b"PAR1" or footer != b"PAR1":
                raise ValueError("Response is not a Parquet file.")

            partial.replace(path)

            incoming = (
                spark.read.parquet(str(path))
                .withColumn(
                    "observed_at_utc",
                    F.expr("try_cast(`DATE` AS TIMESTAMP)")
                )
                .withColumn("snapshot_loaded_at_utc", F.current_timestamp())
                .withColumn("source_url", F.lit(url))
            )

            invalid = incoming.filter(
                F.col("STATION").isNull()
                | (F.col("STATION") != station_id)
                | F.col("observed_at_utc").isNull()
                | (F.year("observed_at_utc") != year)
            ).limit(1).count()

            if invalid:
                raise ValueError("Unexpected station, year, or timestamp.")

            new_stats = incoming.agg(
                F.count("*").alias("n"),
                F.min("observed_at_utc").alias("first"),
                F.max("observed_at_utc").alias("last")
            ).first()

            if new_stats.n == 0:
                raise ValueError("Empty file; keeping existing data.")

            old_stats = spark.table(TARGET).filter(condition).agg(
                F.count("*").alias("n"),
                F.min("observed_at_utc").alias("first"),
                F.max("observed_at_utc").alias("last")
            ).first()

            # A smaller or shorter archive needs review before replacement.
            if old_stats.n and (
                new_stats.n < old_stats.n
                or new_stats.first > old_stats.first
                or new_stats.last < old_stats.last
            ):
                raise ValueError(
                    "Download has fewer records or reduced date coverage; "
                    "keeping existing data for review."
                )

            (
                incoming.write
                .format("delta")
                .mode("overwrite")
                .option("replaceWhere", condition)
                .saveAsTable(TARGET)
            )

            record.update(status="updated", rows=new_stats.n,
                          latest_observation_utc=new_stats.last.isoformat())
            print(
                f"UPDATED {station_id} {year}: "
                f"{new_stats.n:,} rows; latest {new_stats.last}"
            )

        except Exception as error:
            message = f"{station_id} {year}: {error}"
            failures.append(message)
            print("FAILED:", message)
            record.update(status="failed", error=str(error))
        finally:
            record["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
            manifest.append(record)
            temporary_manifest = manifest_path.with_suffix(".part")
            temporary_manifest.write_text(json.dumps(manifest, indent=2))
            temporary_manifest.replace(manifest_path)

if failures:
    raise RuntimeError(
        "Some updates failed. Successful station updates were retained.\n"
        + "\n".join(failures)
    )

## Check mapped stations and fail the job on problems

In [0]:
MAX_OBSERVATION_AGE_HOURS = 96
MAX_REFRESH_AGE_HOURS = 30

# Start with the mapping, so a station missing from the registry/raw table is detected.
active = spark.sql("""
WITH mapped AS (
    SELECT noaa_station_id, COUNT(*) AS mapped_bike_stations
    FROM workspace.default.citibike_noaa_mapping GROUP BY noaa_station_id
), latest AS (
    SELECT STATION AS noaa_station_id,
           MAX(observed_at_utc) AS latest_observation_utc,
           MAX(snapshot_loaded_at_utc) AS latest_refresh_utc
    FROM workspace.default.noaa_weather_raw GROUP BY STATION
)
SELECT m.*, r.download_enabled,
       l.latest_observation_utc, l.latest_refresh_utc,
       (unix_timestamp(current_timestamp()) - unix_timestamp(l.latest_observation_utc)) / 3600.0
           AS observation_age_hours,
       (unix_timestamp(current_timestamp()) - unix_timestamp(l.latest_refresh_utc)) / 3600.0
           AS refresh_age_hours
FROM mapped m
LEFT JOIN workspace.default.noaa_station_registry r USING (noaa_station_id)
LEFT JOIN latest l USING (noaa_station_id)
""")

if active.limit(1).count() == 0:
    raise RuntimeError("No mapped weather stations; cannot pass freshness checks.")

problems = active.filter(
    F.col("download_enabled").isNull()
    | (F.col("download_enabled") == F.lit(False))
    | F.col("latest_observation_utc").isNull()
    | F.col("latest_refresh_utc").isNull()
    | (F.col("observation_age_hours") < -1)
    | (F.col("observation_age_hours") > MAX_OBSERVATION_AGE_HOURS)
    | (F.col("refresh_age_hours") > MAX_REFRESH_AGE_HOURS)
)
display(active.orderBy("noaa_station_id"))
problem_rows = problems.collect()
if problem_rows:
    raise RuntimeError("Mapped-station freshness/configuration check failed: "
                       + str([row.asDict() for row in problem_rows]))
print("Freshness check passed for all mapped weather stations.")